# Worked solutions

This notebook is identical to the student version except that every 🔵 `# TODO` has been filled in, **with commentary on why the answer is what it is** rather than just the code. Read the comments — the reasoning is the point, not the syntax.

Everything else, including the ✏️ YOUR TURN cells, is unchanged: those have no single right answer.


# Module E — Clinical records and epidemiology

## Which life-course factors predict a later dementia diagnosis?

> ⚠️ **Simulated cohort with published effect sizes built in. You are learning the method, not discovering the biology.**

### What you will be able to do by the end

1. read a survival dataset — where the outcome is not just *whether* but *when*
2. draw and interpret a Kaplan–Meier curve, the workhorse plot of clinical epidemiology
3. fit both an epidemiological model (hazard ratios) and a machine-learning model to the same question, and say what each one is for
4. compute **Shapley values** to explain a single person's predicted risk
5. name three ways an observational cohort can produce a confident, wrong conclusion

### The data

**This cohort is simulated.** Individual-level electronic health records are never openly redistributable — there is no version of this dataset we could legally ship you. So 900 participants were generated from a survival model whose hazard ratios are set to *published* population estimates: APOE ε4 roughly 2.6× per allele, diabetes ~1.45×, depression history ~1.6×, more education protective, and so on. The associations you recover should therefore match the literature — because we put them there. What is real is the **method**, and the traps.

### How to work through this notebook

Run the cells in order, top to bottom. The notebook is split into four sections:

| | Section | What happens |
|---|---|---|
| 1 | **Understand the data** | Meet every column and every person in the table |
| 2 | **Quality control** | Find the flaws before they fool you |
| 3 | **Build models** | Start from something trivial, then climb |
| 4 | **Read the results** | Turn numbers into a clinical judgement |

Look out for these markers:

- ✏️ **YOUR TURN** — change the value shown, re-run the cell, watch the figure change. Everyone does these.
- 🟢 run and read · 🔵 write a little code · ⚫ take home
- 🧠 a question to think about; the answer is hidden underneath, so try first

**In a hurry?** Skim section 2, then work through 3.2 (Kaplan–Meier) and section 4 (Shapley values) properly.

---

*Teaching material. Nothing here is a diagnostic tool, and no result in this notebook is clinical evidence.*


In [ ]:
# Run me first. This finds the project folder, loads the shared helpers,
# and prints exactly where this module's data came from.
from pathlib import Path
import sys
repo_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src').exists())
sys.path.insert(0, str(repo_root / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plots
from data import load_data, load_extra, provenance
from models import split_data, train_model, evaluate, compare_models, sweep_parameter, MODEL_CHOICES

pd.set_option('display.width', 160)
print(provenance('E'))


---
# 1 · Understand the data

One row is one person, followed from a baseline visit until one of three things happens: they are diagnosed with dementia, they die, or the study ends. **Which of those three happened matters as much as when.**


### 1.1 The columns

| Column | Meaning |
|---|---|
| `age_baseline` | Age when they joined the study. |
| `sex`, `education_years` | |
| `apoe4_dose` | Number of *APOE* ε4 alleles: 0, 1 or 2. The strongest common genetic risk factor. |
| `hypertension`, `diabetes`, `smoking` | Midlife vascular risk factors — the modifiable ones. |
| `physical_activity` | 1 = regularly active. Protective in most cohorts. |
| `depression_history` | 1 = history of depression. **Read the 🧠 box in 1.3 before you interpret this one.** |
| `baseline_mmse`, `systolic_bp` | Measurements at the baseline visit. |
| `followup_years` | **How long we watched them.** |
| `diagnosis_event` | 1 = diagnosed with dementia during follow-up. |
| `died_without_diagnosis` | 1 = died first. This is a **competing risk** — you cannot be diagnosed after you die. |

The pair (`followup_years`, `diagnosis_event`) is what makes this **survival data**. Someone with `diagnosis_event = 0` and `followup_years = 2.1` is not "a healthy person" — they are "a person we only watched for two years". Treating those two as the same is the classic beginner's error.


In [ ]:
df = load_data('E')
print(f'{len(df)} participants.')
print(f'  diagnosed during follow-up : {int(df.diagnosis_event.sum())}')
print(f'  died without a diagnosis   : {int(df.died_without_diagnosis.sum())}')
print(f'  still undiagnosed at the end: {int(((df.diagnosis_event == 0) & (df.died_without_diagnosis == 0)).sum())}')
print(f'  median follow-up           : {df.followup_years.median():.1f} years\n')
df.head()


### 1.2 Who is in the cohort, and how long did we watch them?


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].hist(df['age_baseline'], bins=25, color='#2c6fbb')
axes[0].set_xlabel('age at baseline (years)'); axes[0].set_ylabel('number of people')
axes[0].set_title('Age when they joined')
axes[1].hist([df.loc[df.diagnosis_event == 1, 'followup_years'],
              df.loc[df.diagnosis_event == 0, 'followup_years']],
             bins=20, stacked=True, color=['#e08214', '#cccccc'],
             label=['diagnosed', 'not diagnosed (censored)'])
axes[1].set_xlabel('years of follow-up'); axes[1].set_ylabel('number of people')
axes[1].set_title('How long each person was observed')
axes[1].legend(fontsize=9)
plt.tight_layout(); plt.show()

print(df.groupby('diagnosis_event')[['age_baseline', 'apoe4_dose', 'education_years', 'followup_years']].mean().round(2))


### 1.3 ✏️ Your turn — one risk factor at a time

For a binary exposure, epidemiology's simplest tool is the **2×2 table** and the **odds ratio**: how many times more likely is a diagnosis among the exposed than the unexposed?


In [ ]:
# ==========================================================================
# ✏️  YOUR TURN
#   Try each of these in turn:
#     'apoe4_dose', 'hypertension', 'diabetes', 'smoking',
#     'physical_activity', 'depression_history', 'sex'
#   Which has the biggest odds ratio? Is that the one you would
#   run a public-health campaign about?
# ==========================================================================
RISK_FACTOR = 'apoe4_dose'

table = pd.crosstab(df[RISK_FACTOR], df['diagnosis_event'])
table.columns = ['no diagnosis', 'diagnosed']
display(table)

rates = 100 * df.groupby(RISK_FACTOR)['diagnosis_event'].mean()
counts = df.groupby(RISK_FACTOR).size()
plots.plot_score_comparison([f'{level}\n(n={counts[level]})' for level in rates.index],
                            rates.tolist(), colours=['#2c6fbb'] * len(rates),
                            title=f'Percentage diagnosed, by {RISK_FACTOR}',
                            ylabel='percent diagnosed')
plt.show()

if table.shape[0] == 2:
    (a, b), (c, d) = table.to_numpy()
    print(f'Odds ratio for {RISK_FACTOR}: {(d * a) / (c * b):.2f}')
    print('An odds ratio of 1.0 means no association. 2.0 means the odds double.')
else:
    print('More than two levels — compare the bars directly, or group them into two.')


🧠 **Think first:** `depression_history` shows a strong association with later dementia. Does treating depression prevent dementia?

<details>
<summary>Click for one good answer</summary>

You cannot tell from this. There are at least three explanations, and the data cannot separate them:

1. **Causation** — depression damages the brain, or reduces the social and cognitive activity that protects it.
2. **Reverse causation** — the earliest changes of dementia, years before diagnosis, *present* as apathy and low mood. The 'risk factor' is actually an early symptom. This is called the **prodromal** period and it is a serious problem for every dementia risk factor measured near the end.
3. **Confounding** — something else (vascular disease, social isolation, inflammation) causes both.

The standard partial fix is a **lag**: exclude everyone diagnosed within, say, five years of the exposure measurement, and see if the association survives. It is a take-home exercise below.

</details>


### 🎚 Go further — pick whichever suits you

- 🟢 **Everyone:** run 1.3 for every risk factor and rank them by odds ratio.
- 🔵 **If you want to write code:** compute odds ratios for all of them in a loop and plot them on a single horizontal bar chart with a line at 1.0. That plot is called a **forest plot** and it is how epidemiology results are published.
- ⚫ **Take home:** look up the 2024 Lancet Commission on dementia prevention. It lists 14 modifiable risk factors and estimates what fraction of cases each could prevent — a calculation called the *population attributable fraction*.


---
# 2 · Quality control

The flaws in a cohort study are rarely missing values. They are structural, and they are the reason epidemiology is a discipline rather than a spreadsheet.


### 2.1 Competing risk — the people who died first

In a cohort with a median age near 70, a substantial number of participants die before they could ever have been diagnosed with dementia. If you count them as "did not get dementia", you will systematically **underestimate** risk in exactly the oldest and sickest people.

Worse: a factor that kills people quickly (heavy smoking, say) can look *protective* against dementia, purely because its victims are not around to be diagnosed. This is a real, documented phenomenon and it has misled published studies.


In [ ]:
outcome = np.where(df.diagnosis_event == 1, 'diagnosed',
                   np.where(df.died_without_diagnosis == 1, 'died first', 'still undiagnosed'))
plots.plot_class_balance(pd.Series(outcome), title='What actually happened to the 900 participants')
plt.show()

band = pd.cut(df['age_baseline'], [54, 65, 72, 80, 95], labels=['55-65', '65-72', '72-80', '80+'])
death_rate = 100 * df.groupby(band, observed=True)['died_without_diagnosis'].mean()
plots.plot_score_comparison(death_rate.index.astype(str).tolist(), death_rate.tolist(),
                            colours=['#c0392b'] * len(death_rate),
                            title='Percentage who died before any dementia diagnosis, by baseline age',
                            ylabel='percent')
plt.show()


### 2.2 ✏️ Your turn — how you treat the dead changes the answer

Three defensible choices, three different numbers.


In [ ]:
# ==========================================================================
# ✏️  YOUR TURN
#   Try all three:
#     'as_healthy' -> count deaths as 'no dementia'  (the naive default)
#     'exclude'    -> drop everyone who died          (also biased, differently)
#     'censor'     -> keep them, but only up to their death (the right answer)
#   Watch the estimated risk move.
# ==========================================================================
HANDLE_DEATHS = 'as_healthy'

if HANDLE_DEATHS == 'exclude':
    working = df[df.died_without_diagnosis == 0].copy()
elif HANDLE_DEATHS == 'censor':
    working = df.copy()   # already censored at death in the followup_years column
else:
    working = df.copy()

overall = 100 * working['diagnosis_event'].mean()
by_age = 100 * working.groupby(pd.cut(working['age_baseline'], [54, 65, 72, 80, 95],
                                      labels=['55-65', '65-72', '72-80', '80+']),
                               observed=True)['diagnosis_event'].mean()

plots.plot_score_comparison(by_age.index.astype(str).tolist(), by_age.tolist(),
                            colours=['#2c6fbb'] * len(by_age), reference=overall,
                            title=f"'{HANDLE_DEATHS}': {overall:.1f}% diagnosed overall (dashed line)",
                            ylabel='percent diagnosed')
plt.show()
print(f'{len(working)} people in this analysis.')
print('Note especially what happens to the OLDEST band under each choice.')


### 2.3 Immortal time

A third trap, and the subtlest. Suppose you define an exposure group by something that takes time to happen — "people who attended at least three follow-up visits", say. To be in that group you must have *survived* long enough to attend three visits. Those months are **immortal time**: by construction, nobody in the group could have had the outcome during them.

The result is a group that looks miraculously protected. This has produced dozens of retracted or corrected findings, including several apparent 'benefits' of medications.


In [ ]:
# A deliberately wrong analysis: define 'long attenders' by their total follow-up.
long_attender = df['followup_years'] > df['followup_years'].median()
rates = 100 * df.groupby(long_attender)['diagnosis_event'].mean()

plots.plot_score_comparison(['short follow-up', 'long follow-up ("good attenders")'],
                            rates.tolist(), colours=['#2c6fbb', '#c0392b'],
                            title='A fabricated "protective effect" of attending for longer',
                            ylabel='percent diagnosed')
plt.show()
print('This bar chart is nonsense, and it is the kind of nonsense that gets published.')
print('Anything that uses follow-up duration to define groups builds the answer into the question.')


### 2.4 QC verdict

**Usable for association, never for causation.** Three standing caveats:

1. Deaths are censored, not counted as healthy. Any absolute risk we quote is a risk *conditional on surviving*.
2. Exposures measured at baseline may be early symptoms rather than causes (see 1.3).
3. Never define a group using anything that happened after baseline.

*(**Express path:** you can start from section 3 — run its catch-up cell first and everything below stands alone.)*


---
# 3 · Two ways to model the same question

Epidemiology and machine learning ask *different questions of the same table*, and confusing them is a common mistake:

| | Epidemiology | Machine learning |
|---|---|---|
| Asks | "Is smoking associated with dementia, and how strongly?" | "For this person, what is the risk?" |
| Wants | An unbiased estimate of one effect, with a confidence interval | The best possible prediction, by any means |
| Fears | Confounding | Overfitting |
| Success | A number you can act on as policy | A score on people it has never seen |

We do both.


### 🚏 Taking the Express path? Run this one cell first

It rebuilds everything sections 3 and 4 need, so you can start here without having run sections 1 and 2 yourself. **If you did run them, run this anyway** — it just redefines the same things and costs a second.


In [ ]:
# Express catch-up: safe to run whether or not you did sections 1 and 2.
df = load_data('E')
print(f'{len(df)} participants; {int(df.diagnosis_event.sum())} diagnosed during follow-up.')
print('Ready for section 3.')


### 3.1 The epidemiologist's plot — Kaplan–Meier

A Kaplan–Meier curve answers: *of the people still undiagnosed at year t, what fraction remain undiagnosed?* It handles the fact that people were watched for different lengths of time, which a simple percentage cannot.

It is computed by walking through time and multiplying survival probabilities at each event. `plots.kaplan_meier` does exactly that in eight readable lines — open `src/plots.py` if you want to see the arithmetic.


In [ ]:
plots.plot_survival(df, 'followup_years', 'diagnosis_event', 'apoe4_dose',
                    title='Time to dementia diagnosis by APOE e4 dose')
plt.show()
print('Curves that separate early and stay separated indicate a strong, sustained effect.')
print('Curves that cross mean the effect changes over time — and a single hazard ratio would hide that.')


### 3.2 ✏️ Your turn — split the curve by anything

Change `SPLIT_BY` and watch the curves separate — or not.


In [ ]:
# ==========================================================================
# ✏️  YOUR TURN
#   Try each of:
#     'apoe4_dose', 'diabetes', 'smoking', 'physical_activity',
#     'depression_history', 'hypertension', 'sex', 'education_group'
#   Which factor separates the curves most? Which one would you
#   rather be able to change about your own life?
# ==========================================================================
SPLIT_BY = 'physical_activity'

working = df.copy()
working['education_group'] = np.where(working['education_years'] >= 13, '13+ years', 'under 13 years')

plots.plot_survival(working, 'followup_years', 'diagnosis_event', SPLIT_BY,
                    title=f'Time to diagnosis, split by {SPLIT_BY}')
plt.show()

for name, group in working.groupby(SPLIT_BY):
    at_five = group[group.followup_years >= 5]
    print(f'  {SPLIT_BY} = {name}: {len(group)} people, {group.diagnosis_event.mean():.1%} diagnosed')


### 3.3 Hazard ratios — the epidemiologist's model

A **Cox proportional-hazards model** estimates, for each factor, how much it multiplies the instantaneous rate of diagnosis, holding the others fixed. That multiplier is the **hazard ratio**: 1.0 = no effect, 2.0 = twice the rate, 0.5 = half.

If `lifelines` is installed we fit a real Cox model. If not, we fall back to a logistic regression, whose odds ratios tell a similar story here. Either way you get the same figure: **a forest plot**.


In [ ]:
predictors = ['age_baseline', 'apoe4_dose', 'education_years', 'hypertension',
              'diabetes', 'smoking', 'physical_activity', 'depression_history']

try:
    from lifelines import CoxPHFitter
    cox_frame = df[predictors + ['followup_years', 'diagnosis_event']].copy()
    cox = CoxPHFitter().fit(cox_frame, duration_col='followup_years', event_col='diagnosis_event')
    ratios = np.exp(cox.params_)
    lower, upper = np.exp(cox.confidence_intervals_.iloc[:, 0]), np.exp(cox.confidence_intervals_.iloc[:, 1])
    kind = 'hazard ratio (Cox model)'
except ImportError:
    from sklearn.linear_model import LogisticRegression
    from sklearn.preprocessing import StandardScaler
    print('lifelines is not installed — using logistic regression odds ratios instead.')
    print('Install with: pip install lifelines\n')
    scaler = StandardScaler().fit(df[predictors])
    fitted = LogisticRegression(max_iter=2000).fit(scaler.transform(df[predictors]), df['diagnosis_event'])
    scale = pd.Series(scaler.scale_, index=predictors)
    ratios = np.exp(pd.Series(fitted.coef_[0], index=predictors) / scale)
    lower = upper = None
    kind = 'odds ratio (logistic regression)'

fig, ax = plt.subplots(figsize=(7.5, 4.2))
positions = np.arange(len(ratios))
ax.scatter(ratios.values, positions, s=70, color='#2c6fbb', zorder=3)
if lower is not None:
    ax.hlines(positions, lower.values, upper.values, color='#2c6fbb', linewidth=2)
ax.axvline(1.0, color='#c0392b', linestyle='--')
ax.set_yticks(positions, ratios.index, fontsize=9)
ax.set_xscale('log')
ax.set_xlabel(f'{kind} — right of the red line means higher risk')
ax.set_title('Forest plot: each factor, adjusted for all the others')
plt.tight_layout(); plt.show()

print(ratios.round(3))


### 3.4 The machine-learning model

Same table, different question: *for a person we have never met, how likely is a diagnosis?* Everything is scored on people the model has not seen.


In [ ]:
X = df[predictors]
y = df['diagnosis_event']
X_train, X_test, y_train, y_test = split_data(X, y)

ladder = ['baseline', 'logistic', 'tree', 'random_forest', 'gradient_boosting']
table = compare_models(ladder, X_train, y_train, X_test, y_test)
display(table)
plots.plot_model_comparison(table, metric='auroc', title='Predicting who will be diagnosed (AUROC)')
plt.show()
print('Expect roughly 0.70-0.80. Risk prediction from ordinary risk factors is genuinely hard,')
print('and anyone claiming 0.95 from data like this has leaked something.')


### 3.5 ✏️ Your turn — one tree you can actually read

A decision tree is a flowchart, and a shallow one can be printed and understood by a clinician. That is worth something: a model nobody can inspect is a model nobody can challenge.


In [ ]:
# ==========================================================================
# ✏️  YOUR TURN
#   Try DEPTH = 1, 2, 3, then 8.
#   At what depth does the flowchart stop being readable?
#   At what depth does the held-out score stop improving?
#   Those two depths are rarely the same, and choosing between them
#   is a real decision, not a technical one.
# ==========================================================================
DEPTH = 3

from sklearn.tree import DecisionTreeClassifier, plot_tree

tree = DecisionTreeClassifier(max_depth=DEPTH, class_weight='balanced', random_state=42)
tree.fit(X_train, y_train)

fig, ax = plt.subplots(figsize=(min(4 + 3 * DEPTH, 20), 3 + 1.6 * DEPTH))
plot_tree(tree, feature_names=list(X_train.columns), class_names=['no diagnosis', 'diagnosed'],
          filled=True, rounded=True, fontsize=8, impurity=False, ax=ax)
ax.set_title(f'A decision tree of depth {DEPTH}')
plt.tight_layout(); plt.show()

swept, train_scores, test_scores = sweep_parameter(
    'tree', 'max_depth', [1, 2, 3, 5, 8, 12], X_train, y_train, X_test, y_test)
plots.plot_parameter_sweep(swept, train_scores, test_scores, 'max_depth',
                           title='Deeper trees memorise. The blue line is what matters.')
plt.show()


### 3.6 🔵 Your turn to write code — the five-year lag

The reverse-causation fix from 1.3. If depression really *causes* dementia, the association should survive when we ignore everyone diagnosed soon after baseline — because those are the people whose 'depression' was most likely an early symptom.

Fill in the `# TODO`.


In [ ]:
# ✅ Worked solution.
LAG_YEARS = 5
early_case = (df['diagnosis_event'] == 1) & (df['followup_years'] < LAG_YEARS)
lagged = df[~early_case].copy()

def odds_ratio(frame, column):
    counts = pd.crosstab(frame[column], frame['diagnosis_event'])
    (a, b), (c, d) = counts.to_numpy()
    return (d * a) / (c * b)

plots.plot_score_comparison(
    [f'all {len(df)} people', f'excluding early cases\n({len(lagged)} people)'],
    [odds_ratio(df, 'depression_history'), odds_ratio(lagged, 'depression_history')],
    colours=['#c0392b', '#2c6fbb'], reference=1.0,
    title='Does the depression association survive a 5-year lag?', ylabel='odds ratio')
plt.show()

# What to expect and how to read it. In THIS dataset the association was generated as a
# genuine causal effect, so it survives the lag almost unchanged — the exposure really did
# come first. In real cohorts the depression-dementia odds ratio typically shrinks under a
# lag, sometimes a lot, which is the signature of reverse causation: part of what looked
# like a risk factor was prodromal disease.
#
# Note the cost. Excluding early cases throws away the events that occurred soonest, which
# are often the most informative ones, so the lagged estimate is less precise. And the lag
# is a judgement call: five years is convention, not physiology. Report the result at
# several lags rather than picking the one you like.


### 🎚 Go further — pick whichever suits you

- 🟢 **Everyone:** in 3.2, find the factor that separates the survival curves most and the one that separates them least.
- 🔵 **If you want to write code:** install `lifelines` (`pip install lifelines`) and re-run 3.3 to get real hazard ratios with confidence intervals, then check whether the intervals cross 1.0.
- ⚫ **Take home:** fit a **Fine–Gray competing-risks model**, which handles death properly rather than censoring it. Compare its estimates with the Cox model's for the oldest participants.


---
# 4 · Read the results

Including the question that matters most for a risk model somebody might actually be shown: **why did it say that about *me*?**


### 4.1 The standard views


In [ ]:
final_model = train_model('gradient_boosting', X_train, y_train)
probability = final_model.predict_proba(X_test)[:, 1]
predicted = (probability >= 0.5).astype(int)
final_metrics = evaluate(final_model, X_test, y_test)

plots.plot_confusion(y_test, predicted, labels=('no diagnosis', 'diagnosed'),
                     title='Held-out participants')
plt.show()
plots.plot_roc_pr(y_test, probability, title='Predicting a future dementia diagnosis')
plt.show()
plots.plot_calibration(y_test, probability)
plt.show()
for name, value in final_metrics.items():
    print(f'  {name:<20s} {value:.3f}')


**Calibration matters more than usual here.** Nobody acts on "you are in the high-risk group"; they act on "your ten-year risk is about 18%". If the model says 18% and the true rate among such people is 40%, the number is not just imprecise — it is misleading in a way that changes decisions about wills, driving, and care.


### 4.2 Shapley values — why did it say that about this person?

A risk score is only usable in a consultation if you can say what drove it. **Shapley values** do exactly that, and they come with a guarantee no other method offers: each person's contributions **add up exactly** to their prediction.

The idea is borrowed from cooperative game theory. Imagine the eight risk factors as players who join a team one at a time; each one's Shapley value is its average contribution to the final prediction, over every possible joining order. With eight features that is all 2⁸ = 256 subsets, which is small enough to compute **exactly** — the widely used `shap` package approximates this because real models have hundreds of features.

⏱ This cell takes a few seconds.


In [ ]:
from interpret import shapley_values, shapley_importance, baseline_prediction

shap_frame = shapley_values(final_model, X_test.head(80), X_train, features=predictors)
average_risk = baseline_prediction(final_model, X_train)

# (a) Globally — which factors move this model's predictions the most?
importance = shapley_importance(shap_frame)
plots.plot_importance(importance.index, importance.values,
                      title='Average influence on predicted dementia risk (exact Shapley values)',
                      xlabel='mean |contribution| to predicted probability')
plt.show()

# (b) A beeswarm-style view: every person, every feature, coloured by whether the value was high.
fig, ax = plt.subplots(figsize=(8, 4.6))
order = importance.index[::-1]
for position, feature in enumerate(order):
    contributions = shap_frame[feature].to_numpy()
    values = X_test.head(80)[feature].to_numpy().astype(float)
    spread = np.linspace(-0.18, 0.18, len(contributions))
    scatter = ax.scatter(contributions, position + spread, c=values, cmap='coolwarm', s=18, alpha=0.85)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_yticks(range(len(order)), order, fontsize=9)
ax.set_xlabel('contribution to this person\'s predicted risk')
ax.set_title('Every held-out person, every factor. Colour = that person\'s value (blue low, red high).')
fig.colorbar(scatter, ax=ax, shrink=0.8, label='feature value')
plt.tight_layout(); plt.show()


### 4.3 ✏️ Your turn — explain one person

This is the figure you would actually put in front of somebody. Change `PERSON` and read their story.


In [ ]:
# ==========================================================================
# ✏️  YOUR TURN
#   Try several values of PERSON (0 to 79). Look for:
#     - somebody the model thought was high risk
#     - somebody it got WRONG (the printout tells you)
#   For each, read the bars as a sentence: 'their risk was pushed up
#   mainly by ___, and pulled down by ___'.
# ==========================================================================
PERSON = 0

contributions = shap_frame.iloc[PERSON].sort_values()
plots.plot_importance(contributions.index, contributions.values,
                      title=f'Person {X_test.index[PERSON]}: what drove their predicted risk',
                      xlabel='contribution to predicted probability (blue = raises risk)')
plt.show()

print('Their actual characteristics:')
print(X_test.iloc[PERSON].to_string())
print()
print(f'  cohort average risk        {average_risk:.3f}')
print(f'  + their contributions      {contributions.sum():+.3f}')
print(f'  = predicted risk           {average_risk + contributions.sum():.3f}')
print(f'  (model actually says       {probability[PERSON]:.3f})')
print(f'\n  What really happened: {"diagnosed" if y_test.iloc[PERSON] == 1 else "not diagnosed"} '
      f'during follow-up.')


🧠 **Think first:** Shapley values tell you what the *model* used. Do they tell you what *causes* dementia?

<details>
<summary>Click for one good answer</summary>

No, and conflating the two is the most common misuse of interpretability tools. A Shapley value says "the model's prediction moved by this much because of this feature". If the model learned a confounded association — say, that people with less education are diagnosed more often, partly because cognitive tests are calibrated on the better educated — then the Shapley plot will faithfully report that confounded association as an important feature. Interpretability makes a model *transparent*, not *correct*. Module D is the antidote.

</details>


### 4.4 Who does this model fail?


In [ ]:
check = X_test.copy()
check['correct'] = (predicted == y_test).astype(int)
check['sex'] = df.loc[X_test.index, 'sex']
check['age_band'] = pd.cut(check['age_baseline'], [54, 65, 72, 80, 95],
                           labels=['55-65', '65-72', '72-80', '80+'])
check['education_group'] = np.where(check['education_years'] >= 13, '13+ years', 'under 13')

for subgroup in ['sex', 'age_band', 'education_group', 'apoe4_dose']:
    plots.plot_subgroup_errors(check.dropna(subset=[subgroup]), subgroup, 'correct',
                               title=f'Proportion correct by {subgroup}')
    plt.show()


### 4.5 Your headline result

For your own notes. No shared scoreboard.


In [ ]:
plots.plot_score_comparison(list(final_metrics), list(final_metrics.values()), reference=0.5,
                            colours=['#2c6fbb'] * 5,
                            title='Module E — gradient boosting on baseline risk factors',
                            ylabel='score')
plt.show()
print(f'Trained on {len(X_train)} participants, tested on {len(X_test)} unseen ones.')
print('Top three factors by Shapley importance:', ', '.join(importance.index[:3]))
for name, value in final_metrics.items():
    print(f'  {name:<20s} {value:.3f}')


### 4.6 What would have to be true before this touched a patient?

1. **The cohort is simulated.** The hazard ratios were put there by us, from published estimates. Nothing here is new evidence about dementia risk.
2. **Association is not causation, and this notebook cannot bridge that gap.** Section 2 shows three distinct ways an observational cohort produces a confident, wrong causal claim.
3. **A risk score is not a diagnosis.** Telling a healthy 68-year-old they have a 20% ten-year risk has real consequences — insurance, employment, how their family treats them — and there is currently no treatment that changes that number much.
4. **Who was recruited?** Cohort studies over-recruit the healthy, the educated and the willing. Risk models built on them systematically misestimate risk for everyone else — and the modifiable risk factors here (hypertension, diabetes, education, physical activity) are precisely the ones distributed most unequally along lines of income and race. A model that quantifies those factors accurately will also reproduce those inequalities in its outputs.

---

### 🧠 Final question for the group discussion

The Lancet Commission estimates roughly 40% of dementia cases are associated with modifiable risk factors. **Given the models you built today, would you rather have a good prediction model, or a good public-health intervention?** What is each one *for*?


### 🎚 Go further — pick whichever suits you

- 🟢 **Everyone:** in 4.3, find a person the model got wrong and read their Shapley plot to see what misled it.
- 🔵 **If you want to write code:** restrict the model to only the *modifiable* factors (drop age and APOE) and see how much predictive power is left. That number is roughly the ceiling on what prevention advice can achieve.
- ⚫ **Take home:** install `shap` and compare its `TreeExplainer` output with our exact values on the same model. How close is the approximation, and how much faster is it?
